## [控制流](https://docs.langchain.com/oss/python/langgraph/graph-api-control-flow)

### 循环结构

In [ ]:
import random

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

@tool
def get_weather(city: str) -> str:
    """获取指定城市天气"""
    if random.random() < 0.5:
        raise ValueError(f"无法获取{city}的天气")
    return f"{city}的天气是晴朗的"

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)
model_with_tools = model.bind_tools([get_weather])

resp = model_with_tools.invoke([HumanMessage(content="北京天气如何")])
print(resp.tool_calls)

#### 一、静态实现

In [ ]:
from typing import Literal
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState


class State(MessagesState):
    user_input: str
    final_output: str

def input_node(state: State):
    return {"messages": [HumanMessage(content=state["user_input"])]}

def llm_node(state: State):
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}

def tool_node(state: State):
    message = state["messages"][-1]
    if not isinstance(message, AIMessage):
        return {}

    tool_messages = []
    for tool_call in message.tool_calls:
        if tool_call["name"] == "get_weather":
            try:
                tool_result = get_weather.invoke(tool_call)
            except Exception as exc:
                # 将工具异常作为工具结果发回模型，让模型决定如何回复
                tool_result = ToolMessage(
                    content=f"天气查询失败：{exc}",
                    tool_call_id=tool_call["id"],
                    name=tool_call["name"],
                )
            tool_messages.append(tool_result)
    return {"messages": tool_messages}

def route(state: MessagesState) -> Literal["tool_node", "output_node"]:
    message = state["messages"][-1]
    if isinstance(message, AIMessage) and message.tool_calls:
        return "tool_node"
    return "output_node"

def output_node(state: State):
    return {"final_output": state["messages"][-1].content}


builder = StateGraph(State)
builder.add_node(input_node)
builder.add_node(llm_node)
builder.add_node(tool_node)
builder.add_node(output_node)
builder.add_edge(START, "input_node")
builder.add_edge("input_node", "llm_node")
builder.add_edge("tool_node", "llm_node")
builder.add_conditional_edges("llm_node", route)
builder.add_edge("output_node", END)
graph = builder.compile()

for event in graph.invoke(
        {
            "user_input": "北京天气如何",
            "messages": [SystemMessage(content="你是一个天气查询助手，如果工具调用失败，尝试重新调用")]
        }, stream_mode="updates"):
    print(event)


In [ ]:
from IPython.display import display

display(graph)

#### 二、动态实现

In [ ]:
from typing import Literal
from langgraph.types import Command
from langchain_core.messages import AIMessage, ToolMessage, SystemMessage
from langgraph.graph import StateGraph, START, END, MessagesState


class State(MessagesState):
    user_input: str
    final_output: str

def input_node(state: State):
    return {"messages": [HumanMessage(content=state["user_input"])]}

def llm_node(state: State) -> Command[Literal["tool_node", "output_node"]]:
    ai_msg = model_with_tools.invoke(state["messages"])
    if ai_msg.tool_calls:
        return Command(
            goto="tool_node",
            update={"messages": [ai_msg]}
        )
    return Command(
        goto="output_node",
        update={"messages": [ai_msg]}
    )

def tool_node(state: State):
    message = state["messages"][-1]
    if not isinstance(message, AIMessage):
        return {}

    tool_messages = []
    for tool_call in message.tool_calls:
        if tool_call["name"] == "get_weather":
            try:
                tool_result = get_weather.invoke(tool_call)
            except Exception as exc:
                # 将工具异常作为工具结果发回模型，让模型决定如何回复
                tool_result = ToolMessage(
                    content=f"天气查询失败：{exc}",
                    tool_call_id=tool_call["id"],
                    name=tool_call["name"],
                )
            tool_messages.append(tool_result)
    return {"messages": tool_messages}

def output_node(state: State):
    return {"final_output": state["messages"][-1].content}


builder = StateGraph(State)
builder.add_node(input_node)
builder.add_node(llm_node)
builder.add_node(tool_node)
builder.add_node(output_node)
builder.add_edge(START, "input_node")
builder.add_edge("input_node", "llm_node")
builder.add_edge("tool_node", "llm_node")
builder.add_edge("output_node", END)
graph = builder.compile()

for event in graph.invoke(
        {
            "user_input": "北京天气如何",
            "messages": [SystemMessage(content="你是一个天气查询助手，如果工具调用失败，尝试重新调用")]
        }, stream_mode="updates"):
    print(event)


In [ ]:
from IPython.display import display

display(graph)